# Generate Layer-Skip Extrapolation Data

This Generate notebook computes and caches the data consumed by `Visualize-layer-skip-diagnostics.ipynb`. The companion Visualize notebook compares layer-skipping effects on next-state extrapolation with the original model's layerwise graph-spectral diagnostics using Llama-3.2-3B, as presented under **Preliminary Layerwise Intervention Analysis** in the appendix.

The sweep evaluates the original model and each of the 28 single-layer bypasses at a fixed context length of $N = 2000$ tokens (1,000 quantized states, each followed by a comma), with 20 realizations per input family. Each comparison uses the same context and held-out next-state target. Predictions use the expected value over numeric tokens `000–999`, and absolute errors are normalized by the quantized-state range, $950 - 50 = 900$, following the next-state extrapolation analysis.

The cache retains predictions and errors for the constant, periodic logistic-map, chaotic logistic-map, and Lorenz inputs. The companion figure pools three families in each of its chaotic and two periodic groups.


Set imports, constants, relative cache paths, sweep axes, and random seeds.


In [ ]:
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import gc
import hashlib
import random
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import torch
from scipy.integrate import solve_ivp
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model and layer-skipping configuration
SEED = 42
ENSEMBLE_SEED = SEED
MODEL_NAME = "meta-llama/Llama-3.2-3B"
NUM_ENSEMBLE_RUNS = 20
NUM_LAYERS = 28

# Fixed context and held-out next-state target
CONTEXT_STATES = 1000
CONTEXT_TOKENS = 2 * CONTEXT_STATES
TOTAL_SEQUENCE_STATES = CONTEXT_STATES + 1

# Quantized-state range and normalization for absolute prediction error
QUANTIZED_STATE_MIN = 50
QUANTIZED_STATE_MAX = 950
ERROR_NORMALIZATION = float(QUANTIZED_STATE_MAX - QUANTIZED_STATE_MIN)
ERROR_LOG_FLOOR = 1e-6

# Include the leading BOS without counting it toward the content-token context
TOKENIZATION_ADD_SPECIAL_TOKENS = True

# Run from this notebook's directory; keep data and checkpoints in cache/
CACHE_PATH = Path("cache/layer_skip_extrapolation_data.npz")
CHECKPOINT_PATH = Path("cache/layer_skip_extrapolation_checkpoint.npz")
# Cache schemas and sweep axes
NPZ_SCHEMA_VERSION = "fixed2000_layer_skip_v2"
CHECKPOINT_SCHEMA_VERSION = "fixed2000_layer_skip_checkpoint_v1"
run_ids = np.arange(NUM_ENSEMBLE_RUNS, dtype=np.int64)
layer_numbers = np.arange(1, NUM_LAYERS + 1, dtype=np.int64)
condition_numbers = np.arange(NUM_LAYERS + 1, dtype=np.int64)  # 0 denotes the original model

# Main numerical input families
data_order = np.asarray([
    "Constant sampled", "Logistic r=3.2", "Logistic r=3.5", "Logistic r=3.55",
    "Logistic r=3.566", "Logistic r=3.5692", "Logistic r=3.5698",
    "Logistic r=3.9", "Logistic r=4.0", "Lorenz",
], dtype=str)
input_classes = np.asarray(["constant"] + ["periodic"] * 6 + ["chaotic"] * 2 + ["lorenz"])

# Seed data generation and model inference
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Define the Lorenz system and extend its sampling grid by one held-out state.


In [ ]:
# Lorenz parameters and sampling grid
LORENZ_SIGMA = 10.0
LORENZ_RHO = 28.0
LORENZ_BETA = 8.0 / 3.0
LORENZ_ARGS = (LORENZ_SIGMA, LORENZ_RHO, LORENZ_BETA)
# Preserve the original sampling interval when adding the next-state target
N = TOTAL_SEQUENCE_STATES
LORENZ_DT = 100.0 / 999.0
lorenz_t_eval = np.arange(N) * LORENZ_DT
LORENZ_T_SPAN = (float(lorenz_t_eval[0]), float(lorenz_t_eval[-1]))


def lorenz(t, y, sigma, rho, beta):
    return [
        sigma * (y[1] - y[0]),
        y[0] * (rho - y[2]) - y[1],
        y[0] * y[1] - beta * y[2],
    ]

Define logistic-map generators, state quantization, and comma-delimited prompt serialization.


In [ ]:
# Periodic logistic-map regimes and burn-in controls
periodic_logistic_periods = {
    3.2: 2,
    3.5: 4,
    3.55: 8,
    3.566: 16,
    3.5692: 32,
    3.5698: 64,
}
logistic_long_n = 20000
logistic_settle_tol = 1e-10


def logistic_sequence(r, x0, n):
    x = np.empty(n, dtype=float)
    x[0] = x0
    for i in range(1, n):
        x[i] = r * x[i - 1] * (1 - x[i - 1])
    return x


def settle_time_float(x, p, tol=1e-10):
    x = np.asarray(x, dtype=float)
    if p is None or p <= 0 or p >= x.size:
        return None
    # Find the first index after which lag-p error remains below tolerance
    err = np.abs(x[p:] - x[:-p])
    suffix_max = np.maximum.accumulate(err[::-1])[::-1]
    hits = np.flatnonzero(suffix_max < tol)
    return int(hits[0]) if hits.size else None


# Scale each trajectory independently into the 050–950 token range
def quantize_to_state_tokens(values):
    values = np.asarray(values, dtype=float)
    value_min = float(values.min())
    value_max = float(values.max())
    if value_max == value_min:
        return np.full(values.shape[0], 500, dtype=int)
    return np.round(
        QUANTIZED_STATE_MIN
        + (values - value_min) / (value_max - value_min) * ERROR_NORMALIZATION
    ).astype(int)


def build_burned_in_logistic_window(
    r,
    x0,
    n_window,
    expected_period,
    n_long=logistic_long_n,
    tol=logistic_settle_tol,
    phase_offset=0,
):
    # Discard transients and select a phase-aligned periodic window
    phase_offset = int(phase_offset) % int(expected_period)
    x_long = logistic_sequence(r, x0, n_long)
    t_settle = settle_time_float(x_long, expected_period, tol=tol)
    if t_settle is None:
        raise ValueError(f"Could not find settle time for r={r} with expected period {expected_period}.")

    start_idx = int(t_settle) + int(phase_offset)
    if start_idx + n_window > x_long.size:
        raise ValueError(f"Need a longer trajectory for r={r}; start index {start_idx} leaves fewer than {n_window} states.")

    return np.asarray(x_long[start_idx:start_idx + n_window], dtype=float)


# Emit one zero-padded number token and one comma token per state
def _serialized_prompt_from_states(states):
    states = np.asarray(states, dtype=int)
    return ",".join(f"{int(x):03d}" for x in states) + ","

Build 20 realizations per input family with shared initial-condition draws and randomized periodic phases.


In [ ]:
# Share each replicate's x0 across all input families
periodic_rs = list(periodic_logistic_periods.keys())
chaotic_rs = [3.9, 4.0]

ensemble_rng = np.random.default_rng(ENSEMBLE_SEED)
phase_rng = np.random.default_rng(ENSEMBLE_SEED + 1)
x0_samples = ensemble_rng.uniform(0.3, 0.7, size=NUM_ENSEMBLE_RUNS)
ensemble_inputs = {}

for rep_id, x0_rep in zip(run_ids, x0_samples):
    x0_rep = float(x0_rep)
    rep_inputs = {}

    # Repeat one sampled token throughout each constant realization
    constant_token = int(
        np.round(QUANTIZED_STATE_MIN + ERROR_NORMALIZATION * x0_rep)
    )
    rep_inputs["Constant sampled"] = np.full(N, constant_token, dtype=int)

    # Use settled periodic windows with randomized phase
    for r in periodic_rs:
        expected_period = periodic_logistic_periods[r]
        phase_offset = int(phase_rng.integers(0, expected_period))
        values = build_burned_in_logistic_window(
            r,
            x0=x0_rep,
            n_window=N,
            expected_period=expected_period,
            phase_offset=phase_offset,
        )
        rep_inputs[f"Logistic r={r}"] = quantize_to_state_tokens(values)

    # Start chaotic trajectories directly from the shared sampled x0
    for r in chaotic_rs:
        values = logistic_sequence(r, x0_rep, N)
        rep_inputs[f"Logistic r={r}"] = quantize_to_state_tokens(values)

    # Perturb Lorenz x0 and quantize the resulting x-coordinate
    lorenz_initial_conditions = [float(-8.0 + (x0_rep - 0.5)), 7.0, 27.0]
    sol = solve_ivp(
        lorenz,
        LORENZ_T_SPAN,
        lorenz_initial_conditions,
        args=LORENZ_ARGS,
        t_eval=lorenz_t_eval,
    )
    if not sol.success:
        raise RuntimeError(f"Lorenz solve failed for rep {int(rep_id)}: {sol.message}")
    rep_inputs["Lorenz"] = quantize_to_state_tokens(sol.y[0])

    missing_inputs = sorted(set(data_order.astype(str)).difference(rep_inputs.keys()))
    if missing_inputs:
        raise ValueError(f"Missing inputs for rep {int(rep_id)}: {missing_inputs}")
    ensemble_inputs[int(rep_id)] = rep_inputs

# Reserve the final state as the target after the 1,000-state context
target_states = np.asarray([[ensemble_inputs[int(rep_id)][str(name)][CONTEXT_STATES]
                             for rep_id in run_ids] for name in data_order], dtype=np.int64)

Load the tokenizer and validate the 2,000-token contexts, excluding the leading BOS.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)


def single_token_id(text):
    ids = tokenizer(text, add_special_tokens=False).input_ids
    if len(ids) != 1:
        raise ValueError(f"Expected one token for {text!r}, got {ids}.")
    return int(ids[0])


# Require one token per number and comma, and map the numeric prediction vocabulary
COMMA_TOKEN_ID = single_token_id(",")
numeric_token_ids = torch.tensor([single_token_id(f"{value:03d}") for value in range(1000)])
numeric_token_values = torch.arange(1000, dtype=torch.float32)


def prepare_fixed_context_inputs(sequence, model):
    sequence = np.asarray(sequence, dtype=int)
    if sequence.shape != (TOTAL_SEQUENCE_STATES,):
        raise ValueError(f"Expected {TOTAL_SEQUENCE_STATES} states, got {sequence.shape}.")
    prompt = _serialized_prompt_from_states(sequence[:CONTEXT_STATES])
    inputs = tokenizer(prompt, add_special_tokens=TOKENIZATION_ADD_SPECIAL_TOKENS,
                       return_tensors="pt").to(next(model.parameters()).device)
    token_ids = inputs.input_ids[0]
    # Exclude the leading BOS when checking the serialized content
    token_start = int(int(token_ids[0]) == tokenizer.bos_token_id)
    content_ids = token_ids[token_start:]
    if content_ids.numel() != CONTEXT_TOKENS or not torch.all(content_ids[1::2] == COMMA_TOKEN_ID):
        raise ValueError("Expected 1,000 number tokens, each followed by one comma.")
    # Record the prompt token ids to validate paired comparisons when resuming
    payload = ",".join(str(int(token)) for token in content_ids.detach().cpu().tolist()).encode("ascii")
    return inputs, hashlib.sha256(payload).hexdigest()

Define single-layer bypassing and the expected next-state value over numeric tokens `000–999`.


In [ ]:
@contextmanager
def skip_decoder_layer(model, layer_number):
    """Replace the chosen decoder block's output with its input; 0 means baseline."""
    if layer_number == 0:
        yield
        return
    if not 1 <= layer_number <= len(model.model.layers):
        raise ValueError(f"Invalid layer number: {layer_number}")

    def return_input(module, args, kwargs, output):
        # Pass the block input to the next layer while retaining the output structure
        hidden_states = args[0] if args else kwargs["hidden_states"]
        return (hidden_states,) + output[1:] if isinstance(output, tuple) else hidden_states

    handle = model.model.layers[layer_number - 1].register_forward_hook(return_input, with_kwargs=True)
    try:
        yield
    finally:
        handle.remove()


def predict_expected_value(inputs, model, layer_number=0):
    with torch.inference_mode(), skip_decoder_layer(model, layer_number):
        outputs = model(**inputs, return_dict=True, use_cache=False)
        logits = outputs.logits[0, -1]
        ids = numeric_token_ids.to(logits.device)
        values = numeric_token_values.to(logits.device)
        # Renormalize over numeric tokens before computing the expected state
        probabilities = torch.softmax(logits.index_select(0, ids).float(), dim=-1)
        prediction = float((probabilities * values).sum().item())
    return prediction

Create, validate, load, and save checkpoint arrays for the resumable layer-skipping sweep.


In [ ]:
def cache_metadata():
    return {
        "model_name": np.asarray(MODEL_NAME),
        "data_order": data_order,
        "run_ids": run_ids,
        "layer_numbers": layer_numbers,
        "fixed_ctx_len_states": np.asarray(CONTEXT_STATES, dtype=np.int64),
        "fixed_seq_len_tokens": np.asarray(CONTEXT_TOKENS, dtype=np.int64),
        "total_sequence_states": np.asarray(TOTAL_SEQUENCE_STATES, dtype=np.int64),
        "fixed_error_normalization": np.asarray(ERROR_NORMALIZATION),
        "fixed_error_log_floor": np.asarray(ERROR_LOG_FLOOR),
        "quantized_state_min": np.asarray(QUANTIZED_STATE_MIN, dtype=np.int64),
        "quantized_state_max": np.asarray(QUANTIZED_STATE_MAX, dtype=np.int64),
        "seed": np.asarray(SEED, dtype=np.int64),
        "ensemble_seed": np.asarray(ENSEMBLE_SEED, dtype=np.int64),
        "num_ensemble_runs": np.asarray(NUM_ENSEMBLE_RUNS, dtype=np.int64),
        "npz_schema_version": np.asarray(NPZ_SCHEMA_VERSION),
    }


def load_or_initialize_checkpoint():
    # Retain one prediction per input family, realization, and model condition
    shape = (len(data_order), len(run_ids), len(condition_numbers))
    predictions = np.full(shape, np.nan, dtype=np.float64)
    prompt_hashes = np.full(shape[:2], "", dtype="U64")
    if not CHECKPOINT_PATH.exists():
        return predictions, prompt_hashes

    with np.load(CHECKPOINT_PATH, allow_pickle=False) as data:
        # Resume only when the model, sweep, targets, and schema match
        expected = {**cache_metadata(), "condition_numbers": condition_numbers,
                    "target_states": target_states,
                    "checkpoint_schema_version": np.asarray(CHECKPOINT_SCHEMA_VERSION)}
        for key, value in expected.items():
            if key not in data.files or not np.array_equal(data[key], value):
                raise ValueError(f"Incompatible checkpoint field: {key}")
        predictions = data["predictions"].astype(np.float64)
        prompt_hashes = data["prompt_token_hashes"].astype(str)
    if predictions.shape != shape or prompt_hashes.shape != shape[:2]:
        raise ValueError("Checkpoint array shapes do not match the configured sweep.")
    if np.isinf(predictions).any():
        raise ValueError("Checkpoint contains infinite predictions.")
    if np.any(np.isfinite(predictions).any(axis=2) & (prompt_hashes == "")):
        raise ValueError("Checkpoint predictions are missing their prompt hashes.")
    return predictions, prompt_hashes


def save_checkpoint(predictions, prompt_hashes):
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    # Write the complete checkpoint before replacing the previous version
    temporary_path = CHECKPOINT_PATH.with_suffix(".tmp.npz")
    np.savez_compressed(
        temporary_path, **cache_metadata(),
        checkpoint_schema_version=np.asarray(CHECKPOINT_SCHEMA_VERSION),
        condition_numbers=condition_numbers, target_states=target_states,
        predictions=predictions, prompt_token_hashes=prompt_hashes,
    )
    temporary_path.replace(CHECKPOINT_PATH)


predictions, prompt_token_hashes = load_or_initialize_checkpoint()

Load the model and validate the decoder-layer count.


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", torch_dtype=torch.float16,
)
model.eval()
if len(model.model.layers) != NUM_LAYERS:
    raise ValueError(f"Expected {NUM_LAYERS} decoder layers, got {len(model.model.layers)}.")

Run or resume the original-model and single-layer-skipping comparisons on identical contexts and targets.


In [ ]:
# Task grid: input family x realization; evaluate the original model and 28 single-layer skips
tasks = [(i, str(name), j, int(rep_id))
         for i, name in enumerate(data_order) for j, rep_id in enumerate(run_ids)]
for input_idx, name, run_idx, rep_id in tqdm(tasks, desc="Layer-skip extrapolation sweep"):
    # Skip input realizations already completed in a compatible checkpoint
    if np.isfinite(predictions[input_idx, run_idx]).all():
        continue
    sequence = ensemble_inputs[rep_id][name]
    inputs, prompt_hash = prepare_fixed_context_inputs(sequence, model)
    previous_hash = prompt_token_hashes[input_idx, run_idx]
    if previous_hash and previous_hash != prompt_hash:
        raise ValueError(f"Prompt changed for {name}, realization {rep_id}.")
    prompt_token_hashes[input_idx, run_idx] = prompt_hash
    # Reuse the same tokenized context for every model condition
    for condition_idx, layer_number in enumerate(condition_numbers):
        if not np.isfinite(predictions[input_idx, run_idx, condition_idx]):
            predictions[input_idx, run_idx, condition_idx] = predict_expected_value(
                inputs, model, int(layer_number)
            )
    # Save after each input realization and release temporary tensors
    save_checkpoint(predictions, prompt_token_hashes)
    del inputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Validate completed predictions and save normalized errors and metadata for the companion visualization.


In [ ]:
if not np.isfinite(predictions).all() or np.any(prompt_token_hashes == ""):
    raise ValueError("Complete all input/realization/condition tasks before exporting.")

# Normalize absolute prediction error by the quantized-state range: 950 - 50 = 900
normalized_abs_error = np.abs(predictions - target_states[:, :, None]) / ERROR_NORMALIZATION
# Flatten input family x realization x model condition into cache columns
input_axis, run_axis, condition_axis = np.meshgrid(
    np.arange(len(data_order)), np.arange(len(run_ids)), np.arange(len(condition_numbers)),
    indexing="ij",
)
skipped_layers = condition_numbers[condition_axis.ravel()]
row_count = predictions.size
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
# Save predictions, targets, errors, and prompt hashes for paired comparisons
np.savez_compressed(
    CACHE_PATH, **cache_metadata(),
    rep_id=run_ids[run_axis.ravel()],
    data_name=data_order[input_axis.ravel()],
    input_class=input_classes[input_axis.ravel()],
    condition=np.where(skipped_layers == 0, "baseline", "skip_layer"),
    skipped_layer_number=skipped_layers,
    ctx_len=np.full(row_count, CONTEXT_STATES, dtype=np.int64),
    seq_len=np.full(row_count, CONTEXT_TOKENS, dtype=np.int64),
    true_next_state=target_states[input_axis.ravel(), run_axis.ravel()],
    predicted_value=predictions.ravel(),
    normalized_abs_error=normalized_abs_error.ravel(),
    prompt_token_hashes=prompt_token_hashes,
    prompt_hash_algorithm=np.asarray("sha256(comma-joined decimal post-BOS token ids)"),
)
print(f"Saved {row_count} predictions to {CACHE_PATH}; error normalization = {ERROR_NORMALIZATION:g}.")